In [11]:
url = "2026410102954_40709_fillable.pdf"

In [ ]:
from pypdf import PdfReader

reader = PdfReader(url)
fields = reader.get_form_text_fields()
fields

{'text_983e_f580': '刘晨',
 'text_7832_9995': '',
 'text_4a76_95d7': '',
 'text_4ce2_f284': '',
 'text_64ad_f4aa': '',
 'textarea_cc00_7ca5': '',
 'text_e5d9_854b': '',
 'text_6c4b_a7b6': '',
 'text_e236_b4f9': '',
 'text_246f_7bba': '',
 'text_a838_fa38': '',
 'text_e870_1b04': '',
 'text_a82e_e695': '',
 'text_6571_ccfe': '',
 'text_c8b7_ec63': '',
 'text_0328_6abb': '',
 'text_4645_0d33': '',
 'text_2aae_cee4': '',
 'text_3d5f_eca1': '',
 'text_a23a_5699': '',
 'text_1388_3793': '',
 'text_b7c6_c194': '34',
 'text_35d0_60c6': '',
 'text_b526_0e3e': '',
 'text_66c2_fc06': '',
 'text_df63_2e44': '',
 'text_27e7_a914': '',
 'text_e99c_67f8': '',
 'text_6e5a_34e1': '',
 'text_aa56_63bc': '',
 'text_1e48_516e': '',
 'text_bced_eaf6': '',
 'text_708b_6c5c': '',
 'text_3159_6be2': '',
 'text_7b61_cb05': '',
 'text_1a30_8f11': '',
 'text_b244_1fc3': '',
 'text_a267_a5df': '',
 'text_8902_ae37': '',
 'text_24ba_1ac0': '',
 'text_ca01_e10c': '',
 'text_ca67_f130': '',
 'textarea_9bec_f212': '',

In [20]:
from typing import Any
from pypdf import PdfReader


def dereference(obj: Any) -> Any:
    """解析 pypdf 的间接对象。"""
    if hasattr(obj, "get_object"):
        return obj.get_object()
    return obj


def get_inherited_value(widget: Any, key: str) -> Any:
    """
    从 Widget 开始向 /Parent 查找字段属性。
    /FT、/T、/V 等属性可能位于 Widget，也可能位于父字段。
    """
    current = widget
    visited = set()

    while current is not None:
        current = dereference(current)

        reference = getattr(current, "indirect_reference", None)
        marker = (
            (reference.idnum, reference.generation)
            if reference is not None
            else id(current)
        )

        if marker in visited:
            break
        visited.add(marker)

        if key in current:
            return current[key]

        parent = current.get("/Parent")
        current = dereference(parent) if parent is not None else None

    return None


def get_qualified_field_name(widget: Any) -> str | None:
    """
    构造完整字段名。
    例如父字段为 applicant，子字段为 name，
    完整字段名为 applicant.name。
    """
    names: list[str] = []
    current = widget
    visited = set()

    while current is not None:
        current = dereference(current)

        reference = getattr(current, "indirect_reference", None)
        marker = (
            (reference.idnum, reference.generation)
            if reference is not None
            else id(current)
        )

        if marker in visited:
            break
        visited.add(marker)

        field_name = current.get("/T")
        if field_name is not None:
            names.append(str(field_name))

        parent = current.get("/Parent")
        current = dereference(parent) if parent is not None else None

    if not names:
        return None

    return ".".join(reversed(names))


def extract_form_widgets(pdf_path: str) -> list[dict[str, Any]]:
    reader = PdfReader(pdf_path)
    result: list[dict[str, Any]] = []

    for page_index, page in enumerate(reader.pages):
        annotations = page.get("/Annots", [])

        for annotation_ref in annotations:
            widget = dereference(annotation_ref)

            if widget.get("/Subtype") != "/Widget":
                continue

            rect = widget.get("/Rect")
            if rect is None:
                continue

            x0, y0, x1, y1 = [float(value) for value in rect]

            field_type = get_inherited_value(widget, "/FT")
            field_value = get_inherited_value(widget, "/V")
            field_name = get_qualified_field_name(widget)

            item = {
                "name": field_name,
                "type": str(field_type) if field_type else None,
                "value": str(field_value) if field_value is not None else None,
                "page": page_index,
                "rect": {
                    "x0": x0,
                    "y0": y0,
                    "x1": x1,
                    "y1": y1,
                    "width": x1 - x0,
                    "height": y1 - y0,
                },
            }

            # 获取复选框、单选按钮的可用状态
            appearance = dereference(widget.get("/AP"))
            if appearance:
                normal_appearance = dereference(appearance.get("/N"))
                if hasattr(normal_appearance, "keys"):
                    item["states"] = [str(key) for key in normal_appearance.keys()]

            result.append(item)

    return result


widgets = extract_form_widgets(url)

for widget in widgets:
    print(widget)

{'name': 'text_983e_f580', 'type': '/Tx', 'value': '刘晨', 'page': 0, 'rect': {'x0': 145.06, 'y0': 750.0, 'x1': 240.42, 'y1': 768.0, 'width': 95.35999999999999, 'height': 18.0}, 'states': ['/BBox', '/Resources', '/Subtype', '/Type', '/Matrix', '/Filter']}
{'name': 'checkbox_e50d_66f3', 'type': '/Btn', 'value': '/Yes', 'page': 0, 'rect': {'x0': 471.18, 'y0': 748.66, 'x1': 491.19, 'y1': 768.0, 'width': 20.00999999999999, 'height': 19.340000000000032}, 'states': ['/Off', '/Yes']}
{'name': 'checkbox_c22c_ee3e', 'type': '/Btn', 'value': '/Off', 'page': 0, 'rect': {'x0': 440.17, 'y0': 749.66, 'x1': 459.51, 'y1': 767.0, 'width': 19.339999999999975, 'height': 17.340000000000032}, 'states': ['/Off', '/Yes']}
{'name': 'checkbox_a7e4_5c1e', 'type': '/Btn', 'value': '/Off', 'page': 0, 'rect': {'x0': 277.11, 'y0': 751.66, 'x1': 286.44, 'y1': 761.0, 'width': 9.329999999999984, 'height': 9.340000000000032}, 'states': ['/Off', '/Yes']}
{'name': 'checkbox_e151_186b', 'type': '/Btn', 'value': '/Yes', 'pag

In [22]:
from pypdf import PdfReader

fields = reader.get_fields() or {}

for full_name, field in fields.items():
    print({
        "full_name": full_name,
        "partial_name": field.get("/T"),
        "type": field.get("/FT"),
        "value": field.get("/V"),
        "has_parent": "/Parent" in field,
        "has_kids": "/Kids" in field,
    })

{'full_name': 'text_983e_f580', 'partial_name': 'text_983e_f580', 'type': '/Tx', 'value': '刘晨', 'has_parent': False, 'has_kids': False}
{'full_name': 'checkbox_e50d_66f3', 'partial_name': 'checkbox_e50d_66f3', 'type': '/Btn', 'value': '/Yes', 'has_parent': False, 'has_kids': False}
{'full_name': 'checkbox_c22c_ee3e', 'partial_name': 'checkbox_c22c_ee3e', 'type': '/Btn', 'value': '/Off', 'has_parent': False, 'has_kids': False}
{'full_name': 'checkbox_a7e4_5c1e', 'partial_name': 'checkbox_a7e4_5c1e', 'type': '/Btn', 'value': '/Off', 'has_parent': False, 'has_kids': False}
{'full_name': 'checkbox_e151_186b', 'partial_name': 'checkbox_e151_186b', 'type': '/Btn', 'value': '/Yes', 'has_parent': False, 'has_kids': False}
{'full_name': 'checkbox_36bc_4442', 'partial_name': 'checkbox_36bc_4442', 'type': '/Btn', 'value': '/Off', 'has_parent': False, 'has_kids': False}
{'full_name': 'checkbox_2ca5_bba3', 'partial_name': 'checkbox_2ca5_bba3', 'type': '/Btn', 'value': '/Off', 'has_parent': False, '

In [15]:
from pypdf import PdfReader, PdfWriter

reader = PdfReader(url)
writer = PdfWriter()

page = reader.pages[0]
fields = reader.get_fields()

writer.append(reader)

writer.update_page_form_field_values(
    writer.pages[0],
    {"text_983e_f580": "笨蛋"},
    auto_regenerate=False,
)

writer.write("out-filled-form.pdf")

(True, <_io.FileIO [closed]>)

In [16]:

fields = reader.get_fields()

In [17]:
fields

{'text_983e_f580': {'/T': 'text_983e_f580',
  '/FT': '/Tx',
  '/Ff': 0,
  '/V': '刘晨'},
 'checkbox_e50d_66f3': {'/T': 'checkbox_e50d_66f3',
  '/FT': '/Btn',
  '/V': '/Yes',
  '/_States_': ['/Off', '/Yes']},
 'checkbox_c22c_ee3e': {'/T': 'checkbox_c22c_ee3e',
  '/FT': '/Btn',
  '/V': '/Off',
  '/_States_': ['/Off', '/Yes']},
 'checkbox_a7e4_5c1e': {'/T': 'checkbox_a7e4_5c1e',
  '/FT': '/Btn',
  '/V': '/Off',
  '/_States_': ['/Off', '/Yes']},
 'checkbox_e151_186b': {'/T': 'checkbox_e151_186b',
  '/FT': '/Btn',
  '/V': '/Yes',
  '/_States_': ['/Off', '/Yes']},
 'checkbox_36bc_4442': {'/T': 'checkbox_36bc_4442',
  '/FT': '/Btn',
  '/V': '/Off',
  '/_States_': ['/Off', '/Yes']},
 'checkbox_2ca5_bba3': {'/T': 'checkbox_2ca5_bba3',
  '/FT': '/Btn',
  '/V': '/Off',
  '/_States_': ['/Off', '/Yes']},
 'checkbox_3cc2_ac51': {'/T': 'checkbox_3cc2_ac51',
  '/FT': '/Btn',
  '/V': '/Off',
  '/_States_': ['/Off', '/Yes']},
 'checkbox_fa0a_d148': {'/T': 'checkbox_fa0a_d148',
  '/FT': '/Btn',
  '/V': '/Y